In [1]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)
import keras_tuner as kt
import pandas as pd

In [2]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [3]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [4]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [6]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [7]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [8]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [9]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

In [11]:
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [12]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [13]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [14]:
basic_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D((2,2)),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
basic_cnn.compile(
    optimizer="SGD",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [16]:
history_basic_improved = basic_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 338ms/step - accuracy: 0.3909 - loss: 1.9257 - val_accuracy: 0.5493 - val_loss: 1.3310 - learning_rate: 0.0100
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 335ms/step - accuracy: 0.4165 - loss: 1.8104 - val_accuracy: 0.1538 - val_loss: 1.7334 - learning_rate: 0.0100
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 327ms/step - accuracy: 0.4315 - loss: 1.6974 - val_accuracy: 0.1791 - val_loss: 1.8061 - learning_rate: 0.0100
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 333ms/step - accuracy: 0.4488 - loss: 1.5731 - val_accuracy: 0.6218 - val_loss: 1.1423 - learning_rate: 0.0100
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 323ms/step - accuracy: 0.4636 - loss: 1.4941 - val_accuracy: 0.5393 - val_loss: 1.1432 - learning_rate: 0.0100
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 325ms/step - accuracy: 0.4555 - loss: 1.4731 - val_accuracy: 0.1738 - val_loss: 1.6773 - learning_rate: 0.0100
Epoch 7/15
219/220 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step - accuracy: 0.48

In [17]:
train_loss, lr_train_acc = basic_cnn.evaluate(train_ds)
valid_loss, lr_valid_acc = basic_cnn.evaluate(valid_ds)
test_loss, lr_test_acc = basic_cnn.evaluate(test_ds)
print(lr_train_acc)
print(lr_valid_acc)
print(lr_test_acc)

220/220 ━━━━━━━━━━━━━━━━━━━━ 25s 105ms/step - accuracy: 0.5471 - loss: 1.0788
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - accuracy: 0.5326 - loss: 1.1126
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - accuracy: 0.5210 - loss: 1.1325
0.547075629234314
0.5326231718063354
0.5209580659866333


In [18]:
results = pd.DataFrame(columns=[
    "Model",
    "Train accuracy",
    "Test accuracy",
    "Valid accuracy"
])
results.loc[len(results)] = [
    "Basic cnn using SGD",
    lr_train_acc,
    lr_test_acc,
    lr_valid_acc
]
results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,Basic cnn using SGD,0.547076,0.520958,0.532623


In [19]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [20]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [21]:
basic_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D((2,2)),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
basic_cnn.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [23]:
history_basic_improved = basic_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 341ms/step - accuracy: 0.3492 - loss: 2.8356 - val_accuracy: 0.4940 - val_loss: 1.3810 - learning_rate: 0.0010
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 339ms/step - accuracy: 0.4225 - loss: 1.6963 - val_accuracy: 0.3968 - val_loss: 1.5666 - learning_rate: 0.0010
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 339ms/step - accuracy: 0.4334 - loss: 1.5842 - val_accuracy: 0.1451 - val_loss: 6.1327 - learning_rate: 0.0010
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step - accuracy: 0.4611 - loss: 1.5751
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 78s 346ms/step - accuracy: 0.4611 - loss: 1.5751 - val_accuracy: 0.3782 - val_loss: 1.5550 - learning_rate: 0.0010
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 341ms/step - accuracy: 0.4983 - loss: 1.3788 - val_accuracy: 0.4474 - val_loss: 1.3220 - learning_rate: 5.0000e-04
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 340ms/step - accuracy: 0

In [24]:
train_loss, rm_train_acc = basic_cnn.evaluate(train_ds)
valid_loss, rm_valid_acc = basic_cnn.evaluate(valid_ds)
test_loss, rm_test_acc = basic_cnn.evaluate(test_ds)
print(rm_train_acc)
print(rm_test_acc)
print(rm_valid_acc)

220/220 ━━━━━━━━━━━━━━━━━━━━ 25s 105ms/step - accuracy: 0.6285 - loss: 0.9019
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - accuracy: 0.6165 - loss: 0.9640
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 112ms/step - accuracy: 0.6094 - loss: 0.9545
0.6285306811332703
0.6094477772712708
0.616511344909668


In [25]:
results.loc[len(results)] = [
    "Basic cnn using RMSprop",
    rm_train_acc,
    rm_test_acc,
    rm_valid_acc
]
results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,Basic cnn using SGD,0.547076,0.520958,0.532623
1,Basic cnn using RMSprop,0.628531,0.609448,0.616511


In [26]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [27]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [28]:
basic_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D((2,2)),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
basic_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [30]:
history_basic_improved = basic_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 351ms/step - accuracy: 0.4148 - loss: 2.1041 - val_accuracy: 0.3083 - val_loss: 1.6503 - learning_rate: 0.0010
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 349ms/step - accuracy: 0.3608 - loss: 1.7367 - val_accuracy: 0.4634 - val_loss: 1.5183 - learning_rate: 0.0010
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 349ms/step - accuracy: 0.4722 - loss: 1.5394 - val_accuracy: 0.2916 - val_loss: 1.8824 - learning_rate: 0.0010
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 358ms/step - accuracy: 0.4401 - loss: 1.5116 - val_accuracy: 0.4487 - val_loss: 1.3546 - learning_rate: 0.0010
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 351ms/step - accuracy: 0.4538 - loss: 1.4474 - val_accuracy: 0.3595 - val_loss: 1.6491 - learning_rate: 0.0010
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 352ms/step - accuracy: 0.4732 - loss: 1.4378 - val_accuracy: 0.1312 - val_loss: 1.9955 - learning_rate: 0.0010
Epoch 7/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 353ms/step - accuracy: 0.4

In [31]:
train_loss, adam_train_acc = basic_cnn.evaluate(train_ds)
valid_loss, adam_valid_acc = basic_cnn.evaluate(valid_ds)
test_loss, adam_test_acc = basic_cnn.evaluate(test_ds)
print(adam_train_acc)
print(adam_test_acc)
print(adam_valid_acc)

220/220 ━━━━━━━━━━━━━━━━━━━━ 26s 109ms/step - accuracy: 0.5993 - loss: 1.0229
47/47 ━━━━━━━━━━━━━━━━━━━━ 6s 117ms/step - accuracy: 0.5692 - loss: 1.0756
47/47 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.5695 - loss: 1.0854
0.5992867350578308
0.5695276260375977
0.5692409873008728


In [32]:
results.loc[len(results)] = [
    "Basic cnn using learning rate and early",
    adam_train_acc,
    adam_test_acc,
    adam_valid_acc
]
results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,Basic cnn using SGD,0.547076,0.520958,0.532623
1,Basic cnn using RMSprop,0.628531,0.609448,0.616511
2,Basic cnn using learning rate and early,0.599287,0.569528,0.569241


In [47]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 16

In [48]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [49]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [50]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [51]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [52]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [53]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [54]:
basic_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D((2,2)),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [55]:
basic_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [56]:
batch_size = 16
history_basic = basic_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    batch_size = batch_size
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


439/439 ━━━━━━━━━━━━━━━━━━━━ 149s 331ms/step - accuracy: 0.4044 - loss: 1.8947 - val_accuracy: 0.1019 - val_loss: 2.0183
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 149s 333ms/step - accuracy: 0.3862 - loss: 1.6749 - val_accuracy: 0.2084 - val_loss: 1.9892
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 147s 329ms/step - accuracy: 0.1997 - loss: 1.7568 - val_accuracy: 0.4441 - val_loss: 1.4454
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 150s 336ms/step - accuracy: 0.4417 - loss: 1.5557 - val_accuracy: 0.2750 - val_loss: 1.6488
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 199s 329ms/step - accuracy: 0.4569 - loss: 1.4923 - val_accuracy: 0.3988 - val_loss: 1.4366
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 201s 327ms/step - accuracy: 0.4894 - loss: 1.4285 - val_accuracy: 0.3722 - val_loss: 1.6864
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 149s 333ms/step - accuracy: 0.4889 - loss: 1.4386 - val_accuracy: 0.4700 - val_loss: 1.4917
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 150s 336ms/step - accuracy: 0.4879 - loss: 1.37

In [57]:
train_loss, bt16_train_acc = basic_cnn.evaluate(train_ds)
valid_loss, bt16_valid_acc = basic_cnn.evaluate(valid_ds)
test_loss, bt16_test_acc = basic_cnn.evaluate(test_ds)
print(bt16_train_acc)
print(bt16_test_acc)
print(bt16_valid_acc)

439/439 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.3351 - loss: 1.7574
94/94 ━━━━━━━━━━━━━━━━━━━━ 8s 88ms/step - accuracy: 0.3322 - loss: 1.7875
94/94 ━━━━━━━━━━━━━━━━━━━━ 8s 87ms/step - accuracy: 0.3240 - loss: 1.7861
0.3350927233695984
0.3240186274051666
0.33222371339797974


In [58]:
results.loc[len(results)] = [
    "Basic cnn using batchsize_16",
    bt16_train_acc,
    bt16_test_acc,
    bt16_valid_acc
]
results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,Basic cnn using SGD,0.547076,0.520958,0.532623
1,Basic cnn using RMSprop,0.628531,0.609448,0.616511
2,Basic cnn using learning rate and early,0.599287,0.569528,0.569241
3,Basic cnn using batchsize_16,0.043509,0.038589,0.037949
4,Basic cnn using batchsize_16,0.335093,0.324019,0.332224


In [59]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 64

In [60]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [61]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [62]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [63]:
basic_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D((2,2)),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [64]:
basic_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [65]:
batch_size = 64
history_basic = basic_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    batch_size = batch_size
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - accuracy: 0.4261 - loss: 2.2518 - val_accuracy: 0.2330 - val_loss: 1.8299
Epoch 2/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.4031 - loss: 1.6569 - val_accuracy: 0.3921 - val_loss: 1.4844
Epoch 3/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.4158 - loss: 1.6767 - val_accuracy: 0.0732 - val_loss: 2.1759
Epoch 4/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 76s 671ms/step - accuracy: 0.3773 - loss: 1.5382 - val_accuracy: 0.2783 - val_loss: 1.7633
Epoch 5/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 75s 659ms/step - accuracy: 0.4551 - loss: 1.4794 - val_accuracy: 0.3808 - val_loss: 1.5706
Epoch 6/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 76s 671ms/step - accuracy: 0.4770 - loss: 1.3996 - val_accuracy: 0.4960 - val_loss: 1.2725
Epoch 7/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 76s 668ms/step - accuracy: 0.5044 - loss: 1.3413 - val_accuracy: 0.3415 - val_loss: 1.7005
Epoch 8/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 75s 666ms/step - accuracy: 0.4642 - loss: 1.4302 - val_accur

In [66]:
train_loss, bt64_train_acc = basic_cnn.evaluate(train_ds)
valid_loss, bt64_valid_acc = basic_cnn.evaluate(valid_ds)
test_loss, bt64_test_acc = basic_cnn.evaluate(test_ds)
print(bt64_train_acc)
print(bt64_test_acc)
print(bt64_valid_acc)

110/110 ━━━━━━━━━━━━━━━━━━━━ 24s 206ms/step - accuracy: 0.5552 - loss: 1.1314
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 208ms/step - accuracy: 0.5340 - loss: 1.2189
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 206ms/step - accuracy: 0.5389 - loss: 1.2086
0.555206835269928
0.538922131061554
0.5339547395706177


In [67]:
results.loc[len(results)] = [
    "Basic cnn using batchsize_64",
    bt64_train_acc,
    bt64_test_acc,
    bt64_valid_acc
]
results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,Basic cnn using SGD,0.547076,0.520958,0.532623
1,Basic cnn using RMSprop,0.628531,0.609448,0.616511
2,Basic cnn using learning rate and early,0.599287,0.569528,0.569241
3,Basic cnn using batchsize_16,0.043509,0.038589,0.037949
4,Basic cnn using batchsize_16,0.335093,0.324019,0.332224
5,Basic cnn using batchsize_64,0.555207,0.538922,0.533955


In [12]:
def build_basic_cnn(hp):
    model = tf.keras.Sequential([
        Conv2D(
            hp.Int(
                "conv1",
                16,
                64,
                step=16
            ),
            (3,3),
            activation="relu",
            input_shape=INPUT_SHAPE
        ),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(
            hp.Int(
                "dense_units",
                64,
                256,
                step=64
            ),
            activation="relu"
        ),
        Dropout(
            hp.Float(
                "dropout",
                0.2,
                0.5,
                step=0.1
            )
        ),
        Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])
    model.compile(
        optimizer=hp.Choice(
            "optimizer",
            [
                "adam",
                "sgd",
                "rmsprop"
            ]
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [13]:
basic_tuner = kt.RandomSearch(
    build_basic_cnn,
    objective="val_accuracy",
    max_trials=5,
    directory="tuning",
    project_name="basic_cnn"
)
basic_tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

Reloading Tuner from tuning/basic_cnn/tuner0.json


In [15]:
best_hps = basic_tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'conv1': 16, 'dense_units': 128, 'dropout': 0.2, 'optimizer': 'adam'}


In [17]:
best_basic = basic_tuner.get_best_models(1)[0]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [75]:
train_loss, hype_train_acc = basic_cnn.evaluate(train_ds)
valid_loss, hype_valid_acc = basic_cnn.evaluate(valid_ds)
test_loss, hype_test_acc = basic_cnn.evaluate(test_ds)
print(hype_train_acc)
print(hype_test_acc)
print(hype_valid_acc)

110/110 ━━━━━━━━━━━━━━━━━━━━ 25s 208ms/step - accuracy: 0.5538 - loss: 1.1342
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 211ms/step - accuracy: 0.5446 - loss: 1.2036
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 217ms/step - accuracy: 0.5316 - loss: 1.2160
0.5537803173065186
0.5316034555435181
0.5446071624755859


In [76]:
results.loc[len(results)] = [
    "Basic cnn hyperparameter",
    hype_train_acc,
    hype_test_acc,
    hype_valid_acc
]
results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,Basic cnn using SGD,0.547076,0.520958,0.532623
1,Basic cnn using RMSprop,0.628531,0.609448,0.616511
2,Basic cnn using learning rate and early,0.599287,0.569528,0.569241
3,Basic cnn using batchsize_16,0.043509,0.038589,0.037949
4,Basic cnn using batchsize_16,0.335093,0.324019,0.332224
5,Basic cnn using batchsize_64,0.555207,0.538922,0.533955
6,Basic cnn hyperparameter,0.553780,0.531603,0.544607


In [77]:
results.to_csv("basic_comparison.csv",index=False)

In [30]:
best_basic.save("basic_cnn_phase5.keras")